# Esoteric Language Models: Bridging Autoregressive and Masked Diffusion LLMs

This notebook implements the core training methodology for **Eso-LMs**. It is heavily optimized to train dynamically on an **RTX 4060 8GB VRAM GPU** by leveraging streaming datasets (0GB disk overhead), `torch.amp` mixed precision, gradient accumulation, and constrained parameter scaling (~45M params).

In [1]:
# Run this once if you don't have the libraries installed:
# !pip install torch transformers datasets

import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from datasets import load_dataset

: 

### 1. Hardware, Precision, and Architecture Definitions

In [ ]:
# Hardware and Precision
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = True

# Model Architecture (Scaled to fit within 8GB VRAM)
VOCAB_SIZE = 50258 # GPT2 vocab + 1 custom [MASK] token
D_MODEL = 768
N_HEAD = 12
NUM_LAYERS = 6
MAX_SEQ_LEN = 256

# Training Params
BATCH_SIZE = 8
ACCUMULATION_STEPS = 4
LEARNING_RATE = 1e-4
MAX_STEPS = 5000

### 2. Dataset and Tokenization
We faithfully use OpenWebText. To save memory, it's streamed.

In [ ]:
# Initialize standard GPT-2 Tokenizer and inject the [MASK] token required for diffusion
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.add_special_tokens({'mask_token': '[MASK]'})
MASK_TOKEN_ID = tokenizer.mask_token_id

# Stream OpenWebText dynamically
dataset = load_dataset("Skylion007/openwebtext", split="train", streaming=True)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"], 
        truncation=True, 
        max_length=MAX_SEQ_LEN, 
        padding="max_length"
    )

tokenized_dataset = dataset.map(tokenize_batch, batched=True, remove_columns=["text"])

def collate_fn(batch):
    # Note: Attention logic/causality is handled entirely via the Eso-Masking algorithm below.
    input_ids = torch.tensor([b["input_ids"] for b in batch], dtype=torch.long)
    return input_ids

dataloader = DataLoader(tokenized_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)

### 3. The Mathematical Core - Vectorized Esoteric Masking
This is the core algorithmic contribution of the paper. It structures the attention matrix on the fly depending on whether the generation step is acting as standard diffusion or AR sequence continuation.

In [ ]:
def create_eso_mask_vectorized(mask_indices, nheads, phase="diffusion"):
    """
    Constructs the exact sequence topologies required by Eso-LMs.
    """
    B, L = mask_indices.shape
    device = mask_indices.device
    
    # Initialize full mask (True means blocked/cannot attend)
    attn_mask = torch.zeros(B, L, L, dtype=torch.bool, device=device)
    
    for b in range(B):
        # 1. Separate clean and masked token indices
        clean_idx = (~mask_indices[b]).nonzero(as_tuple=True)[0]
        mask_idx = mask_indices[b].nonzero(as_tuple=True)[0]
        
        # 2. Clean tokens are always randomly ordered among themselves
        clean_perm = clean_idx[torch.randperm(len(clean_idx), device=device)]
        
        # 3. Mask tokens ordering depends on the generative phase
        if phase == "diffusion":
            mask_perm = mask_idx[torch.randperm(len(mask_idx), device=device)]
        else: # "ar" sequential phase
            mask_perm = mask_idx 
            
        # Combine to create the permutation sigma
        sigma = torch.cat([clean_perm, mask_perm])
        
        # 4. Vectorized mask creation based on permutation ranks
        # If token y comes after token x in sigma, x cannot attend to y.
        inv_sigma = torch.empty_like(sigma)
        inv_sigma[sigma] = torch.arange(L, device=device)
        
        ranks = inv_sigma.unsqueeze(1)    # (L, 1) -> query ranks
        ranks_T = inv_sigma.unsqueeze(0)  # (1, L) -> key ranks
        
        # Query at rank `i` cannot attend to Key at rank `j` if j > i
        attn_mask[b] = ranks_T > ranks

    # Expand to shape expected by PyTorch MultiheadAttention: (B * nheads, L, L)
    attn_mask = attn_mask.unsqueeze(1).expand(B, nheads, L, L).reshape(B * nheads, L, L)
    return attn_mask

### 4. Eso-LM Model Implementation

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=4096):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class EsoTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, max_seq_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_seq_len)
        
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=d_model*4, 
            dropout=0.1, 
            batch_first=True, 
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layers, num_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.nhead = nhead
        
    def forward(self, x, mask_indices, phase="diffusion"):
        # Generate the dynamic permutation mask
        attn_mask = create_eso_mask_vectorized(mask_indices, self.nhead, phase=phase)
        
        emb = self.embedding(x)
        emb = self.pos_encoder(emb)
        
        # Process through transformer utilizing the esoteric constraint mask
        out = self.transformer(emb, mask=attn_mask)
        logits = self.fc_out(out)
        return logits

model = EsoTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    nhead=N_HEAD,
    num_layers=NUM_LAYERS,
    max_seq_len=MAX_SEQ_LEN
).to(DEVICE)

print(f"Model instantiated! Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")

### 5. Training Loop: Negative Evidence Lower Bound (NELBO) 
We train by randomly interchanging the diffusion and AR mask topologies in real time while tracking the cross-entropy loss isolated on masked tokens.

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scaler = torch.amp.GradScaler('cuda') if USE_AMP and torch.cuda.is_available() else None

model.train()
step = 0
iterator = iter(dataloader)

print("Starting Eso-LM Training...")

while step < MAX_STEPS:
    optimizer.zero_grad()
    
    for _ in range(ACCUMULATION_STEPS):
        try:
            input_ids = next(iterator)
        except StopIteration:
            iterator = iter(dataloader)
            input_ids = next(iterator)
            
        input_ids = input_ids.to(DEVICE)
        B, L = input_ids.shape
        
        # 1. Masking Schedule: Sample alpha_t from Uniform(0, 1)
        alpha_t = torch.rand(B, 1, device=DEVICE)
        
        # Create boolean mask indices (True means replace with [MASK])
        rand_probs = torch.rand(B, L, device=DEVICE)
        mask_indices = rand_probs > alpha_t
        
        # Create the continuous-time noisy sequence z_t
        z_t = input_ids.clone()
        z_t[mask_indices] = MASK_TOKEN_ID
        
        # 2. Stochastically choose objective (50% Diffusion Phase, 50% AR Phase)
        phase = "diffusion" if torch.rand(1).item() < 0.5 else "ar"
        
        if USE_AMP and torch.cuda.is_available():
            with torch.amp.autocast('cuda'):
                logits = model(z_t, mask_indices, phase=phase)
                
                # 3. NELBO Loss calculation
                loss_fct = nn.CrossEntropyLoss(reduction='none')
                loss = loss_fct(logits.view(-1, VOCAB_SIZE), input_ids.view(-1)).view(B, L)
                
                # Calculate loss only over dynamically masked targets
                masked_loss = (loss * mask_indices.float()).sum() / (mask_indices.float().sum() + 1e-6)
                masked_loss = masked_loss / ACCUMULATION_STEPS
                
            # Accumulate scaled gradients    
            scaler.scale(masked_loss).backward()
        else:
            logits = model(z_t, mask_indices, phase=phase)
            loss_fct = nn.CrossEntropyLoss(reduction='none')
            loss = loss_fct(logits.view(-1, VOCAB_SIZE), input_ids.view(-1)).view(B, L)
            masked_loss = (loss * mask_indices.float()).sum() / (mask_indices.float().sum() + 1e-6)
            masked_loss = masked_loss / ACCUMULATION_STEPS
            masked_loss.backward()
        
    if USE_AMP and torch.cuda.is_available():
        scaler.step(optimizer)
        scaler.update()
    else:
        optimizer.step()
        
    step += 1
    
    if step % 50 == 0:
        print(f"Step {step:4d} | Running NELBO Loss: {masked_loss.item() * ACCUMULATION_STEPS:.4f} | Final Acc Phase: {phase}")

print("Training Complete!")